# 🗄️ Esportazione Automatica da SQL Server a Excel

Questo script Python automatizza l'estrazione completa dei dati da un database **SQL Server**, esportando ogni tabella in un file Excel dedicato (`.xlsx`). 

È progettato per essere robusto e gestire automaticamente conversioni di dati complessi e saltare le tabelle di sistema non necessarie.

## 🚀 Caratteristiche Principali

* **Connessione Sicura:** Utilizza l'autenticazione di Windows tramite driver ODBC 17.
* **Recupero Dinamico:** Ottiene la lista delle tabelle tramite una Stored Procedure dedicata (`sp_ListaTabelle`).
* **Sanificazione Dati:** Identifica i campi contenenti dati binari (es. immagini, file) e li converte nella stringa di testo `<BINARY DATA>` per evitare crash durante il salvataggio.
* **Formattazione Excel:** Tronca automaticamente il nome del file a 31 caratteri per rispettare i limiti intrinseci dei fogli di calcolo Excel.
* **Gestione Errori:** Se l'esportazione di una tabella fallisce, lo script non si interrompe ma continua con la tabella successiva.

## 🛠️ Requisiti e Installazione

Prima di eseguire lo script, assicurati di avere installato il driver **ODBC Driver 17 for SQL Server** sul tuo sistema e di installare le librerie Python necessarie. 

Puoi installare le dipendenze eseguendo questo comando nel terminale:

```bash
pip install pandas pyodbc sqlalchemy openpyxl matplotlib

In [1]:
# --- IMPORTAZIONE DELLE LIBRERIE ---
import tkinter as tk
import urllib.parse
from sqlalchemy import create_engine
import pyodbc  # Libreria fondamentale per connettersi a database ODBC (come SQL Server)
from tkinter import ttk, filedialog, messagebox
import matplotlib.pyplot as plt
import pandas as pd  # Libreria per l'analisi dei dati, essenziale per trasformare i dati in Excel
import numpy as np
import openpyxl  # Motore che Pandas usa dietro le quinte per scrivere i file .xlsx
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

# --- CONFIGURAZIONE DELLA CONNESSIONE ---
# Qui definiamo la stringa di connessione. Specifica i driver, il server, il database 
# e indica di usare l'autenticazione di Windows (Trusted_Connection=yes).
conn_str_windows = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=Cristian_Pico\\SQLEXPRESS;"          # Nome dell'istanza SQL Server
    "DATABASE=ScuolaDb;"                         # Nome del database a cui collegarsi
    "Trusted_Connection=yes;"                    # Usa le credenziali dell'utente Windows attuale
    "TrustServerCertificate=yes;"                # Evita errori legati ai certificati SSL locali
)

# Iniziamo un blocco "try...except" per gestire eventuali errori critici (es. server spento)
try:
    print("\n🥲  Connessione al server...")

    # CREIAMO LA CONNESSIONE PYODBC (Ci serve per il Cursore)
    conn = pyodbc.connect(conn_str_windows)
    cursor = conn.cursor()

    # Codifica la stringa in un formato leggibile per SQLAlchemy
    params = urllib.parse.quote_plus(conn_str_windows)
    # Crea il "motore"
    engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
    
    # Esegue una piccola query per farsi restituire il nome del server a cui ci siamo collegati
    nome_server = cursor.execute('SELECT @@SERVERNAME').fetchone()[0]

    print(f"\n😎 Connesso al server {nome_server}!")
    print("\nRecupera automaticamente tutte le tabelle\n")

    # Esegue una Stored Procedure (creata in precedenza nel DB) per ottenere l'elenco delle tabelle
    cursor.execute("EXEC dbo.sp_ListaTabelle")
    
    # Estrae il primo elemento (row[0]) di ogni riga restituita e crea una lista di nomi di tabelle
    tabelle = [row[0] for row in cursor.fetchall()]

    # Stampa a video l'elenco delle tabelle trovate
    print("|Tabelle trovate|\n")
    for t in tabelle:
        print(f"-> {t}")

    print("\nEsportazione delle tabelle in file Excel in corso...\n")
    
    # --- CICLO DI ESPORTAZIONE ---
    # Passa in rassegna ogni singola tabella trovata nel database
    for tabella in tabelle:
        
        # Le tabelle di sistema non ci interessano, quindi le ignoriamo
        if tabella.lower() in ["sysdiagrams", "sysdiagram"]:
            print("⏭️  Salto la tabella di sistema: sysdiagrams")
            continue  # Interrompe questo giro del ciclo e passa alla tabella successiva

        print(f"Esportazione: {tabella}")

        # Un blocco "try...except" interno: se l'esportazione di una singola tabella fallisce, 
        # il programma non si blocca ma passa alla successiva.
        try:
            # Prepara la query per estrarre tutti i dati dalla tabella corrente
            query = f"SELECT * FROM [{tabella}]"
            
            # Pandas esegue la query e salva tutti i dati nella variabile 'df' (DataFrame)
            df = pd.read_sql_query(query, engine)

            # --- PULIZIA DEI DATI BINARI ---
            # Excel non supporta il salvataggio di dati binari "grezzi" (come immagini o file salvati nel DB).
            # Dobbiamo trasformarli in testo normale prima di esportare.
            for col in df.columns:
                # Controlla se la colonna contiene testo o oggetti complessi
                if df[col].dtype == object:
                    # Applica una funzione a ogni cella della colonna:
                    # Se il dato è di tipo "bytes" (binario), scrive "<BINARY DATA>", altrimenti lascia il dato originale
                    df[col] = df[col].apply(
                        lambda x: "<BINARY DATA>" if isinstance(x, bytes) else x
                    )

            # --- SALVATAGGIO IN EXCEL ---
            # Taglia il nome del file a 31 caratteri. Questo è utile perché Excel non accetta
            # nomi di fogli (sheet) più lunghi di 31 caratteri.
            file_name = f"{tabella[:31]}.xlsx"
            
            # Crea effettivamente il file Excel. index=False evita di salvare la colonna coi numeri di riga (0, 1, 2...)
            df.to_excel(file_name, index=False)
            
            print(f"✅ Salvata nel foglio: {file_name}\n")

        except Exception as ex:
            # Se la singola query o il salvataggio falliscono, stampa il motivo dell'errore
            print(f"❌ Errore nella tabella {tabella}: {ex}\n")

    # Se il ciclo finisce senza che il programma "esploda", stampa questo messaggio
    print("✅ Esportazione di tutte le tabelle completata con successo!")

# Questo blocco cattura errori generali o problemi di connessione iniziale
except Exception as e:
    print("❌ Errore di connessione o errore fatale:")
    print(e)

# Il blocco "finally" viene eseguito SEMPRE, sia che ci siano stati errori, sia che sia andato tutto bene
finally:
    try:
        # È fondamentale chiudere la connessione al database per non lasciare sessioni "appese" nel server
        conn.close()
        engine.dispose()
        print("\n🫡  Connessione al server disconnessa con successo.\n")
    except:
        # Se fallisce anche la chiusura, stampa un messaggio generico
        print("Errore durante la chiusura della connessione...😒")


🥲  Connessione al server...

😎 Connesso al server Cristian_Pico\SQLEXPRESS!

Recupera automaticamente tutte le tabelle

|Tabelle trovate|

-> sysdiagrams
-> Studenti
-> Corsi
-> Docenti
-> Aule
-> Iscrizioni
-> Lezioni
-> DocentiCorso

Esportazione delle tabelle in file Excel in corso...

⏭️  Salto la tabella di sistema: sysdiagrams
Esportazione: Studenti
✅ Salvata nel foglio: Studenti.xlsx

Esportazione: Corsi
✅ Salvata nel foglio: Corsi.xlsx

Esportazione: Docenti
✅ Salvata nel foglio: Docenti.xlsx

Esportazione: Aule
✅ Salvata nel foglio: Aule.xlsx

Esportazione: Iscrizioni
✅ Salvata nel foglio: Iscrizioni.xlsx

Esportazione: Lezioni
✅ Salvata nel foglio: Lezioni.xlsx

Esportazione: DocentiCorso
✅ Salvata nel foglio: DocentiCorso.xlsx

✅ Esportazione di tutte le tabelle completata con successo!

🫡  Connessione al server disconnessa con successo.



## RECUPERO TABELLE

In [ ]:
def carico_tabella():
    conn = strCon()

    if conn in None:
        return "⚠️ Attenzione verifica che il collegamento al Database ci sia"

    cursor = conn.cursor()

    cursor.execute("EXEC sp_ListaTabelle")

    tabella = [row[0] for row in cursor.fetchall()]
    print("|Tabelle trovate|\n")
    for t in tabella:
        print(f"-> {t}")
        print()
    conn.close()